# Temperature Downscaling: Random Forest vs XGBoost vs CatBoost vs MLP vs CNN

**Question:** given coarse temperature + location + time, can a model predict temperature on a finer grid?

**Setup (proxy):** the NMC dataset is 2.5°. We coarsen it to 5° as the *input* and use the original 2.5° as the *target*.
When ERA5 + a DEM + a real fine target are ready, only sections 3, 5 and 6 change.

**Pipeline:** load -> coarse input / fine target -> features -> spatial split -> interpolation baseline + 5 models -> comparison

## 1. Install

In [ ]:
!pip install xarray netcdf4 xgboost catboost -q

## 2. Imports

In [ ]:
import time as timer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

seed = 42
epochs = 20
torch.manual_seed(seed)
np.random.seed(seed)

## 3. Load data

In [ ]:
ds = xr.open_dataset("/content/air_temperature.nc")
print(ds)

## 4. Quick exploration

In [ ]:
ds["air"].isel(time=0).plot()
plt.title(f"Temperature - {ds.time.values[0]}")
plt.show()

In [ ]:
point = ds["air"].sel(lat=40, lon=260, method="nearest")
point.isel(time=slice(0, 120)).plot()
plt.title("Temperature at lat 40, lon 260 (first 30 days)")
plt.show()

## 5. Coarse input and fine target

In [ ]:
fine = ds["air"] - 273.15                                     # target: 2.5 deg grid, Celsius
coarse = fine.coarsen(lat=2, lon=2, boundary="trim").mean()   # input: 5 deg grid

# coarse grid interpolated back onto the fine grid = the non-ML baseline
coarse_up = coarse.interp(lat=fine.lat, lon=fine.lon, method="linear",
                          kwargs={"fill_value": "extrapolate"})

print(fine.shape, coarse.shape, coarse_up.shape)

## 6. Features table

Target = **residual** (`fine - coarse_up`), not the raw temperature. Trees can't extrapolate and output piecewise-constant values, so letting them predict only the correction on top of the interpolation works much better. Final prediction = `coarse_up + predicted residual`.

Features: interpolated coarse temp, lat, lon, month, hour, day of year. (With real data, add elevation, land cover, humidity, wind, etc. here.)

In [ ]:
data = xr.Dataset({"fine": fine, "coarse_up": coarse_up})
df = data.to_dataframe().reset_index()

df["month"] = df["time"].dt.month
df["hour"] = df["time"].dt.hour
df["dayofyear"] = df["time"].dt.dayofyear
df["residual"] = df["fine"] - df["coarse_up"]

print(df.shape)
df.head()

## 7. Spatial train/test split

Neighbouring grid points are almost identical, so a random split leaks. We cut the map into 5x5-cell blocks and hold out ~20% of whole blocks as the test set.

In [ ]:
lats = np.sort(df["lat"].unique())
lons = np.sort(df["lon"].unique())

df["lat_i"] = np.searchsorted(lats, df["lat"])
df["lon_i"] = np.searchsorted(lons, df["lon"])
df["block"] = (df["lat_i"] // 5) * 100 + (df["lon_i"] // 5)

rng = np.random.default_rng(42)
blocks = df["block"].unique()
test_blocks = rng.choice(blocks, size=int(0.2 * len(blocks)), replace=False)

is_test = df["block"].isin(test_blocks)
train = df[~is_test]
test = df[is_test]

print("train rows:", len(train), "| test rows:", len(test))
print("test blocks:", len(test_blocks), "of", len(blocks))

## 8. Feature matrices

Random subsample of the training rows keeps Random Forest fast. Test set is kept whole.

In [ ]:
features = ["coarse_up", "lat", "lon", "month", "hour", "dayofyear"]

train_s = train.sample(n=200_000, random_state=42)

X_train = train_s[features]
y_train = train_s["residual"]

X_test = test[features]
y_test = test["residual"]
fine_test = test["fine"]
base_test = test["coarse_up"]

## 9. Baseline: interpolation only
The score every model has to beat.

In [ ]:
results = []

pred_fine = base_test.values
results.append({
    "Model": "Interpolation (baseline)",
    "RMSE": np.sqrt(mean_squared_error(fine_test, pred_fine)),
    "MAE": mean_absolute_error(fine_test, pred_fine),
    "R2": r2_score(fine_test, pred_fine),
    "Train time (s)": 0.0,
})
baseline_rmse = results[0]["RMSE"]
pd.DataFrame(results)

## 10. Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=50, n_jobs=-1, random_state=42)

t0 = timer.time()
rf.fit(X_train, y_train)
rf_time = timer.time() - t0

rf_pred = base_test.values + rf.predict(X_test)

results.append({
    "Model": "Random Forest",
    "RMSE": np.sqrt(mean_squared_error(fine_test, rf_pred)),
    "MAE": mean_absolute_error(fine_test, rf_pred),
    "R2": r2_score(fine_test, rf_pred),
    "Train time (s)": rf_time,
})
pd.DataFrame(results)

## 11. XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.1,
                   subsample=0.8, colsample_bytree=0.8,
                   tree_method="hist", n_jobs=-1, random_state=42)

t0 = timer.time()
xgb.fit(X_train, y_train)
xgb_time = timer.time() - t0

xgb_pred = base_test.values + xgb.predict(X_test)

results.append({
    "Model": "XGBoost",
    "RMSE": np.sqrt(mean_squared_error(fine_test, xgb_pred)),
    "MAE": mean_absolute_error(fine_test, xgb_pred),
    "R2": r2_score(fine_test, xgb_pred),
    "Train time (s)": xgb_time,
})
pd.DataFrame(results)

## 12. CatBoost

In [ ]:
cat = CatBoostRegressor(iterations=300, depth=8, learning_rate=0.1,
                        random_seed=42, verbose=0)

t0 = timer.time()
cat.fit(X_train, y_train)
cat_time = timer.time() - t0

cat_pred = base_test.values + cat.predict(X_test)

results.append({
    "Model": "CatBoost",
    "RMSE": np.sqrt(mean_squared_error(fine_test, cat_pred)),
    "MAE": mean_absolute_error(fine_test, cat_pred),
    "R2": r2_score(fine_test, cat_pred),
    "Train time (s)": cat_time,
})
pd.DataFrame(results)

## 13. MLP

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

y_std = y_train.std()

Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor((y_train / y_std).values, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_test_s, dtype=torch.float32)

loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=1024, shuffle=True)

print(device, Xtr.shape, ytr.shape, Xte.shape)

In [ ]:
mlp = nn.Sequential(
    nn.Linear(6, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1),
).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-2)

losses = []

t0 = timer.time()
for epoch in range(epochs):
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    losses.append(total / len(Xtr))
    print(f"epoch {epoch+1}/{epochs}  loss {losses[-1]:.4f}")
mlp_time = timer.time() - t0

print("train time (s):", round(mlp_time, 1))

In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_res = mlp(Xte.to(device)).cpu().numpy().ravel() * y_std

mlp_pred = base_test.values + mlp_res

results.append({
    "Model": "MLP (PyTorch)",
    "RMSE": np.sqrt(mean_squared_error(fine_test, mlp_pred)),
    "MAE": mean_absolute_error(fine_test, mlp_pred),
    "R2": r2_score(fine_test, mlp_pred),
    "Train time (s)": mlp_time,
})

pd.DataFrame(results)

## 14. CNN

In [ ]:
X = coarse_up.values.astype("float32")             # input map: (time, lat, lon)
y = (fine - coarse_up).values.astype("float32")    # target: residual map

# mark which pixels belong to the held-out blocks (same blocks as before)
lat_i = np.searchsorted(lats, fine.lat.values)
lon_i = np.searchsorted(lons, fine.lon.values)
block = (lat_i // 5)[:, None] * 100 + (lon_i // 5)[None, :]
test_mask = np.isin(block, test_blocks)

# scale
X = (X - X.mean()) / X.std()
cnn_std = y[:, ~test_mask].std()
y = y / cnn_std

# to tensors, shape (time, 1, lat, lon)
X = torch.tensor(X).unsqueeze(1)
y = torch.tensor(y).unsqueeze(1)
mask = torch.tensor(~test_mask, dtype=torch.float32)[None, None]   # 1 = train pixel

cnn_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

print(X.shape, y.shape, mask.shape, "test pixels:", int(test_mask.sum()))

In [ ]:
torch.manual_seed(42)

cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
    nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
    nn.Conv2d(32, 1, 3, padding=1),
).to(device)

optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
mask_d = mask.to(device)

epochs = 20

t0 = timer.time()
for epoch in range(epochs):
    total = 0
    for xb, yb in cnn_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = cnn(xb)
        loss = (((out - yb) ** 2) * mask_d).sum() / (mask_d.sum() * len(xb))
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    print(f"epoch {epoch+1}/{epochs}  loss {total / len(X):.4f}")
cnn_time = timer.time() - t0

print("train time (s):", round(cnn_time, 1))

In [ ]:
cnn.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X), 500):
        preds.append(cnn(X[i:i+500].to(device)).cpu())
cnn_res = torch.cat(preds).squeeze(1).numpy() * cnn_std     # residual maps: (time, lat, lon)

# pick the CNN prediction for every row of the test table
ti = fine.get_index("time").get_indexer(test["time"])
yi = fine.get_index("lat").get_indexer(test["lat"])
xi = fine.get_index("lon").get_indexer(test["lon"])
cnn_pred = base_test.values + cnn_res[ti, yi, xi]

results.append({
    "Model": "CNN (PyTorch)",
    "RMSE": np.sqrt(mean_squared_error(fine_test, cnn_pred)),
    "MAE": mean_absolute_error(fine_test, cnn_pred),
    "R2": r2_score(fine_test, cnn_pred),
    "Train time (s)": cnn_time,
})

pd.DataFrame(results)

## 15. Final comparison

In [ ]:
comp = pd.DataFrame(results).drop_duplicates("Model", keep="last")   # safe if a cell is re-run
comp["Improvement vs baseline (%)"] = (1 - comp["RMSE"] / baseline_rmse) * 100
comp = comp.sort_values("RMSE").reset_index(drop=True)
comp.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

ax[0].bar(comp["Model"], comp["RMSE"], color="steelblue")
ax[0].set_title("RMSE (lower is better)")

ax[1].bar(comp["Model"], comp["MAE"], color="seagreen")
ax[1].set_title("MAE (lower is better)")

ax[2].bar(comp["Model"], comp["Train time (s)"], color="indianred")
ax[2].set_title("Train time (s)")

for a in ax:
    a.tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.show()

## 16. Feature importance (tree models)

In [ ]:
imp = pd.DataFrame({
    "Random Forest": rf.feature_importances_,
    "XGBoost": xgb.feature_importances_,
    "CatBoost": cat.feature_importances_ / 100,
}, index=features)

imp.plot(kind="bar", figsize=(8, 4))
plt.title("Feature importance (each model's own scale)")
plt.tight_layout()
plt.show()

## 17. Where does the best model fail? (error map on held-out blocks)

In [ ]:
best_name = comp.loc[comp["Model"] != "Interpolation (baseline)", "Model"].iloc[0]
best_pred = {"Random Forest": rf_pred, "XGBoost": xgb_pred, "CatBoost": cat_pred,
             "MLP (PyTorch)": mlp_pred, "CNN (PyTorch)": cnn_pred}[best_name]
print("Best model:", best_name)

err = test[["lat", "lon"]].copy()
err["abs_error"] = np.abs(fine_test.values - best_pred)
err_map = err.groupby(["lat", "lon"])["abs_error"].mean().reset_index()

plt.figure(figsize=(10, 4))
sc = plt.scatter(err_map["lon"], err_map["lat"], c=err_map["abs_error"], cmap="magma", s=60)
plt.colorbar(sc, label="Mean abs error (C)")
plt.title(f"{best_name}: mean absolute error on held-out blocks")
plt.xlabel("lon"); plt.ylabel("lat")
plt.show()

## 18. Visual comparison: coarse vs downscaled vs truth (one time step)

Maps cover the full grid, so training blocks are included here. The error numbers above are the honest test result.

In [ ]:
t_idx = 1000
t = fine.time.values[t_idx]
one = df[df["time"] == t].sort_values(["lat", "lon"])

if best_name == "MLP (PyTorch)":
    xo = torch.tensor(scaler.transform(one[features]), dtype=torch.float32)
    mlp.eval()
    with torch.no_grad():
        res = mlp(xo.to(device)).cpu().numpy().ravel() * y_std
elif best_name == "CNN (PyTorch)":
    yi1 = fine.get_index("lat").get_indexer(one["lat"])
    xi1 = fine.get_index("lon").get_indexer(one["lon"])
    res = cnn_res[t_idx][yi1, xi1]
else:
    best_model = {"Random Forest": rf, "XGBoost": xgb, "CatBoost": cat}[best_name]
    res = best_model.predict(one[features])

down = one["coarse_up"].values + res

shape = (len(lats), len(lons))
truth_map = one["fine"].values.reshape(shape)
coarse_map = one["coarse_up"].values.reshape(shape)
down_map = down.reshape(shape)

vmin, vmax = truth_map.min(), truth_map.max()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, m, title_ in zip(ax, [coarse_map, down_map, truth_map],
                        ["Coarse (interpolated)", f"Downscaled ({best_name})", "Truth (fine)"]):
    im = a.imshow(m, origin="lower", vmin=vmin, vmax=vmax, cmap="coolwarm")
    a.set_title(title_)
fig.colorbar(im, ax=ax, label="C")
plt.show()

## 19. Caveats (read before quoting the comparison)
- **Proxy data.** Input and target both come from the same 2.5° NMC grid, with no elevation or other fine-scale features. Gains over interpolation are therefore small; rankings may change on ERA5 + DEM.
- **Unequal data budgets.** The tree models and the MLP train on a 200k-row subsample. The CNN trains on every time step of the training blocks.
- **Unequal features.** The CNN sees only the coarse map (no lat, lon or time features); the other models see all six features.
- **One split, one seed.** Differences of a few hundredths in RMSE are within noise. Repeat with different held-out blocks and seeds before claiming a winner.
- **Same test set for every model:** the held-out blocks from section 7.